# 🦠 02: Kanser Kök Hücreleri (CSC) ve THY1/CD90 Biyobelirteç Analizi

Bu analizde:
1. Kolorektal kanser ve stromal hücre ekspresyon profilleri
2. THY1/CD90 ekspresyonunun kemorezistans ile korelasyonu
3. Diferansiyel ekspresyon (Volcano Plot) analizi
4. Random Forest ile minimal biyobelirteç panelinin süzülmesi modellenir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.scrna_pipeline import generate_synthetic_stem_cell_data, ScRNAPipeline
from src.biomarker_discovery import DifferentialExpression, BiomarkerClassifier

counts_df, meta_df = generate_synthetic_stem_cell_data(n_cells=500, seed=42)
pipe = ScRNAPipeline()
log1p_df = pipe.preprocess(counts_df)
print("Veri hazırlandı!")

In [ ]:
# Diferansiyel Ekspresyon (CSC vs Somatik Fibroblast)
csc_cells = meta_df[meta_df["cell_type"] == "Kolorektal CSC"].index.tolist()
fibro_cells = meta_df[meta_df["cell_type"] == "Somatik Fibroblast"].index.tolist()

de_results = DifferentialExpression.analyze(log1p_df, csc_cells, fibro_cells)
print("En çok artan ilk 5 gen:")
print(de_results.head(5)[["gene", "log2_fold_change", "fdr_q_value", "status"]])

In [ ]:
# Makine Öğrenmesi ile Biyobelirteç Seçimi
sub_idx = csc_cells + fibro_cells
X = log1p_df.loc[sub_idx]
y = pd.Series([1 if i in csc_cells else 0 for i in sub_idx], index=sub_idx)

clf = BiomarkerClassifier(n_estimators=50, random_state=42)
importances = clf.fit_and_rank(X, y, top_k=5)
eval_res = clf.evaluate_cv(X, y, n_splits=5)

print(f"ROC-AUC Başarısı: {eval_res['roc_auc']:.3f}")
print("Seçilen en ayırt edici 5 biyobelirteç:", clf.selected_biomarkers)